<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# NLP I: Language Data Pre-Processing and Sentiment Analysis

_Authors: Matt Brems, Noelle Brown_

---

### Learning Objectives

1. Define and implement **text normalizing**: tokenizing, lemmatizing, and stemming.
1. Define and apply sentiment analysis.

#### Before we begin, try running this:

In [1]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
lemmatizer.lemmatize("avocados")

'avocado'

In [2]:
# if you get an error with the above code, run this & follow below directions:
# NOTE: You may need to restart your Jupyter kernel before it works!
# nltk.download()

In [3]:
# Imports
import numpy as np
import pandas as pd       
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [4]:
# Define spam text.
spam = 'Hello,\nI saw your contact information on LinkedIn. I have carefully read through your profile and you seem to have an outstanding personality. This is one major reason why I am in contact with you. My name is Mr. Valery Grayfer Chairman of the Board of Directors of PJSC "LUKOIL". I am 86 years old and I was diagnosed with cancer 2 years ago. I will be going in for an operation later this week. I decided to WILL/Donate the sum of 8,750,000.00 Euros(Eight Million Seven Hundred And Fifty Thousand Euros Only etc. etc.'

print(spam)

Hello,
I saw your contact information on LinkedIn. I have carefully read through your profile and you seem to have an outstanding personality. This is one major reason why I am in contact with you. My name is Mr. Valery Grayfer Chairman of the Board of Directors of PJSC "LUKOIL". I am 86 years old and I was diagnosed with cancer 2 years ago. I will be going in for an operation later this week. I decided to WILL/Donate the sum of 8,750,000.00 Euros(Eight Million Seven Hundred And Fifty Thousand Euros Only etc. etc.


# Text Normalizing

When dealing with text data, there are common pre-processing steps. We won't necessarily use all of them every time we deal with text data.

- Remove special characters
- Stop word removal
- Tokenizing
- Lemmatizing/Stemming

## Removing special characters & Tokenizing

We need to remove unnecessary characters when cleaning text data (punctuation, symbols, etc.). This can be done with a custom regex tokenizer.

When we "**tokenize**" data, we take it and split it up into distinct chunks based on some pattern.

In [5]:
sent_tokenize(spam.lower())

['hello,\ni saw your contact information on linkedin.',
 'i have carefully read through your profile and you seem to have an outstanding personality.',
 'this is one major reason why i am in contact with you.',
 'my name is mr. valery grayfer chairman of the board of directors of pjsc "lukoil".',
 'i am 86 years old and i was diagnosed with cancer 2 years ago.',
 'i will be going in for an operation later this week.',
 'i decided to will/donate the sum of 8,750,000.00 euros(eight million seven hundred and fifty thousand euros only etc.',
 'etc.']

In [6]:
word_tokenize(spam.lower())[:10]

['hello',
 ',',
 'i',
 'saw',
 'your',
 'contact',
 'information',
 'on',
 'linkedin',
 '.']

In [7]:
# Instantiate Tokenizer
tokenizer = RegexpTokenizer('\w+')

In [8]:
# "Run" Tokenizer
spam_tokens = tokenizer.tokenize(spam.lower())

In [9]:
# Show Results
print(spam_tokens)

['hello', 'i', 'saw', 'your', 'contact', 'information', 'on', 'linkedin', 'i', 'have', 'carefully', 'read', 'through', 'your', 'profile', 'and', 'you', 'seem', 'to', 'have', 'an', 'outstanding', 'personality', 'this', 'is', 'one', 'major', 'reason', 'why', 'i', 'am', 'in', 'contact', 'with', 'you', 'my', 'name', 'is', 'mr', 'valery', 'grayfer', 'chairman', 'of', 'the', 'board', 'of', 'directors', 'of', 'pjsc', 'lukoil', 'i', 'am', '86', 'years', 'old', 'and', 'i', 'was', 'diagnosed', 'with', 'cancer', '2', 'years', 'ago', 'i', 'will', 'be', 'going', 'in', 'for', 'an', 'operation', 'later', 'this', 'week', 'i', 'decided', 'to', 'will', 'donate', 'the', 'sum', 'of', '8', '750', '000', '00', 'euros', 'eight', 'million', 'seven', 'hundred', 'and', 'fifty', 'thousand', 'euros', 'only', 'etc', 'etc']


<details><summary>In comparing the original text to our tokenized version of the text, we converted one long string into a list of strings. What other changes occurred?</summary>

- All strings were converted to lower case.
- All punctuation was removed. (This was done using **regular expressions**.)
</details>

## Stop Word Removal

The following quote has had stop words (and punctuation) removed:

"Answer great question life universe everything said deep thought said deep thought paused forty two said deep thought infinite majesty calm."

<details><summary>What book is the above sentence from?</summary>

The Hitchhiker's Guide to the Galaxy!
    
![](../images/hgg.jpg)
    
The original quote reads:  
..."The Answer to the Great Question..."  
"Yes..!"  
"Of Life, the Universe and Everything..." said Deep Thought.  
"Yes...!"  
"Is..." said Deep Thought, and paused.  
"Yes...!"  
"Is..."  
"Yes...!!!...?"  
"Forty-two," said Deep Thought, with infinite majesty and calm.”
</details>

<details><summary>If you were familiar with the book, how did you know what book the sentence was from?</summary>

Removing stop words did not remove key identifying words such as "life", "universe", "everything", and "forty-two".
</details>

<details><summary>Based on this, how would you define stop words?</summary>

Stop words are words that have little to no significance or meaning. They are common words that only add to the grammatical structure and flow of the sentence, so it is still relatively easy to identify the contents of sentences without stop words.
</details>

In [10]:
# What are the stopwords, anyway?
# Note that this list isn't objective! There are other sources with other lists.
print(stopwords.words("english"))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [11]:
# Remove stopwords from "spam_tokens."
spam_tokens_ns = [token for token in spam_tokens if token not in stopwords.words('english')]

In [12]:
# Check it
print(spam_tokens_ns)

['hello', 'saw', 'contact', 'information', 'linkedin', 'carefully', 'read', 'profile', 'seem', 'outstanding', 'personality', 'one', 'major', 'reason', 'contact', 'name', 'mr', 'valery', 'grayfer', 'chairman', 'board', 'directors', 'pjsc', 'lukoil', '86', 'years', 'old', 'diagnosed', 'cancer', '2', 'years', 'ago', 'going', 'operation', 'later', 'week', 'decided', 'donate', 'sum', '8', '750', '000', '00', 'euros', 'eight', 'million', 'seven', 'hundred', 'fifty', 'thousand', 'euros', 'etc', 'etc']


# Stemmin' and Lemmin'
Stemming and lemmatizing are both techniques for reducing words to their base forms. The key differences are...

**Stemming** simply trims suffixes off words according to a few simple rules. It's considered faster but less reliable. For example:

* Jumping -> Jump
* Jumps -> Jump
* Jumped -> Jump

**Lemmatizing** is smart enough to conjugate irregular verbs. There are many more rules so it's considerably slower than stemming. In addition, you must also know the part of speech of the word you're lemmatizing. It can handle:

* Gone -> Go
* Went -> Go
* Going -> Go

### Stemming with the Porter Stemmer
The most common stemmer is the Porter stemmer. Let's play with it and see what it does to our spam email.

In [13]:
ps = PorterStemmer()

In [14]:
ps.stem('jumping')

'jump'

In [15]:
ps.stem('restaurant')

'restaur'

In [16]:
ps.stem('going')

'go'

In [17]:
ps.stem('went')

'went'

In [18]:
# Try some of your own out here!
ps.stem('superlative')

'superl'

In [19]:
' '.join([ps.stem(token) for token in spam_tokens_ns])

'hello saw contact inform linkedin care read profil seem outstand person one major reason contact name mr valeri grayfer chairman board director pjsc lukoil 86 year old diagnos cancer 2 year ago go oper later week decid donat sum 8 750 000 00 euro eight million seven hundr fifti thousand euro etc etc'

### Lemmatizing
The most common lemmatizer is the WordNet lemmatizer. However in order to use it, you also need to enter the part of speech for your word. This can be tricky, as **POS taggers** use the full document to tag a token, and we haven't done that. So we could POS tag what we have so far, but we might get weird results.

In [20]:
nltk.pos_tag(['wound'])

[('wound', 'NN')]

In [21]:
wn = WordNetLemmatizer()

In [22]:
wn.lemmatize('going', 'v')

'go'

In [23]:
wn.lemmatize('went', 'v')

'go'

In [24]:
wn.lemmatize('gone', 'v')

'go'

In [25]:
wn.lemmatize('are', 'v')

'be'

In [26]:
wn.lemmatize('cats', 'n')

'cat'

In [27]:
# Try out your own here!
wn.lemmatize('lovely', wordnet.ADV)

'lovely'

In [28]:
# We can use nltk.pos_tag to (attempt to) tag each token.
nltk.pos_tag(spam_tokens_ns)[:10]

[('hello', 'NN'),
 ('saw', 'VBD'),
 ('contact', 'JJ'),
 ('information', 'NN'),
 ('linkedin', 'NN'),
 ('carefully', 'RB'),
 ('read', 'VB'),
 ('profile', 'NN'),
 ('seem', 'VBP'),
 ('outstanding', 'JJ')]

In [29]:
# For some reason, the WordNet lemmatizers don't match up with what NLTK likes to give us,
# so here's a function that can translate them easily for us.
def custom_lemmatize(word, tag):
    mapper = {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV
    }
    pos = mapper.get(word)
    
    return wn.lemmatize(word, pos) if pos else word

In [30]:
' '.join([custom_lemmatize(word, tag) for word, tag in nltk.pos_tag(spam_tokens_ns)])

'hello saw contact information linkedin carefully read profile seem outstanding personality one major reason contact name mr valery grayfer chairman board directors pjsc lukoil 86 years old diagnosed cancer 2 years ago going operation later week decided donate sum 8 750 000 00 euros eight million seven hundred fifty thousand euros etc etc'

---

# Sentiment Analysis

![](../images/sent.jpeg)

[Sentiment analysis](https://www.kdnuggets.com/2018/08/emotion-sentiment-analysis-practitioners-guide-nlp-5.html) is an area of natural language processing in which we seek to classify text as having positive or negative emotion.

Let's build a simple function that can classify text as either having positive or negative sentiment.

What words tell us whether certain text is positive?

In [31]:
# Let's come up with a list of positive and negative words we might observe.

positive_words = ['delight', 'good', 'great', 'awesome', 'tremendous', 'fabulous', 'amazing', 'stellar']
negative_words = ['garbage', 'sad', 'trash', 'ugly', 'bad', 'disgusting', 'terrible', 'gross']

In [32]:
def simple_sentiment(text):
    # Normalize input text
    tokenizer = RegexpTokenizer(r'\w+')  
    tokens = tokenizer.tokenize(text.lower()) 
    ps = PorterStemmer()
    stemmed_words = [ps.stem(i) for i in tokens]
    
    # Stem the pos and neg vocab words
    positive_stems = [ps.stem(i) for i in positive_words]
    negative_stems = [ps.stem(i) for i in negative_words]

    # Count "positive" and "negative" sentimented words.
    pos_count = len([i for i in stemmed_words if i in positive_stems])
    neg_count = len([i for i in stemmed_words if i in negative_stems])
    
    # Calculate some sentiment
    return (pos_count - neg_count) / (pos_count + neg_count + 1)

In [33]:
# Run our sentiment analyzer on our spam email.
simple_sentiment(spam)

0.0

In [34]:
# Three not-so-random Chipotle reviews.

yelp_1 = "No Chipotle should ever have a 2 out of 5 star rating on Yelp. Especially not this one. As a regular (usually two or three visits a week), I have never been dissatisfied with a single meal here. It's Chipotle, so you know you'll pay $8 (after tax) for a chicken bowl and be full and satisfied afterwards. \n The employees are friendly and give generous portions. Seating is limited, but there is a place you can stand and eat near the window, which is where I always eat. I'm sitting down eight hours a day at the office anyway - standing and eating here is probably extending my lifespan. \nThe line gets line long during lunch, but it moves fast. Dinner time is amazing - rarely a line and the portions are extra generous during this time.\n This fairly new Chipotle is at a great location, near McPherson Square. It's right next to my office and gym so it's perfect for me. \nBottom line: if you're craving Chipotle and are worried about the other reviews and low ratings for this location, don't be. It's my favorite Chipotle location in the DC area, and that's not an exaggeration."

yelp_2 = "DISGUSTING LONG HAIR THREADED THROUGH CHICKEN IN BURRITO BOWL.\n There was a long blonde hair threaded through my chicken as I was eating a burrito bowl.  I did not notice until it was too late and the HAIR ENTERED MY MOUTH as I was eating and I grossly pulled the hair out.\n I calmly walked up to the register to inform them that there was hair in my food. The register person was busy, I understand that, but I was promptly ignored like my issue was not a big deal.  He proceeded to get his manager, 'Leslie' I believe.  She was not apologetic at all and offered no condolences. She did however offer a refund, but I didn't care about the money, I just wanted to eat food without eating someone's hair as a side dish.\n The second time I went back up, a different person, the general manager Peris, was more apologetic and handled the situation better. He ended up getting Leslie to file a report, but who knows if they submitted it or not.\n Suffice to say, if you dont want food in your hair, dont eat here."

yelp_3 = "First time going to this Chipotle.  The line was very quick and the food was fresh.  But as I started eating a notice that the food was very salty.  I started separating my bowl after two bites.  I ordered a bowl with white rice, black beans, chicken, sour cream, cheese and lettuce.  I tasted everything separately.  Once I tasted the Chicken by it self it was unbearable.  It taste like someone pouring the entire bottle of salt on tge chicken.  I tried to take most the chicken out the bowl but still I could not bear the taste of the salt.  So I ended up throwing the damn bowl away.  $8.00 down the drain.  SMH."

In [35]:
yelp_1

"No Chipotle should ever have a 2 out of 5 star rating on Yelp. Especially not this one. As a regular (usually two or three visits a week), I have never been dissatisfied with a single meal here. It's Chipotle, so you know you'll pay $8 (after tax) for a chicken bowl and be full and satisfied afterwards. \n The employees are friendly and give generous portions. Seating is limited, but there is a place you can stand and eat near the window, which is where I always eat. I'm sitting down eight hours a day at the office anyway - standing and eating here is probably extending my lifespan. \nThe line gets line long during lunch, but it moves fast. Dinner time is amazing - rarely a line and the portions are extra generous during this time.\n This fairly new Chipotle is at a great location, near McPherson Square. It's right next to my office and gym so it's perfect for me. \nBottom line: if you're craving Chipotle and are worried about the other reviews and low ratings for this location, don't

In [36]:
yelp_2

"DISGUSTING LONG HAIR THREADED THROUGH CHICKEN IN BURRITO BOWL.\n There was a long blonde hair threaded through my chicken as I was eating a burrito bowl.  I did not notice until it was too late and the HAIR ENTERED MY MOUTH as I was eating and I grossly pulled the hair out.\n I calmly walked up to the register to inform them that there was hair in my food. The register person was busy, I understand that, but I was promptly ignored like my issue was not a big deal.  He proceeded to get his manager, 'Leslie' I believe.  She was not apologetic at all and offered no condolences. She did however offer a refund, but I didn't care about the money, I just wanted to eat food without eating someone's hair as a side dish.\n The second time I went back up, a different person, the general manager Peris, was more apologetic and handled the situation better. He ended up getting Leslie to file a report, but who knows if they submitted it or not.\n Suffice to say, if you dont want food in your hair, d

In [37]:
yelp_3

'First time going to this Chipotle.  The line was very quick and the food was fresh.  But as I started eating a notice that the food was very salty.  I started separating my bowl after two bites.  I ordered a bowl with white rice, black beans, chicken, sour cream, cheese and lettuce.  I tasted everything separately.  Once I tasted the Chicken by it self it was unbearable.  It taste like someone pouring the entire bottle of salt on tge chicken.  I tried to take most the chicken out the bowl but still I could not bear the taste of the salt.  So I ended up throwing the damn bowl away.  $8.00 down the drain.  SMH.'

In [38]:
# Calculate sentiment of yelp_1.
simple_sentiment(yelp_1)

0.6666666666666666

In [39]:
# Calculate sentiment of yelp_2.
simple_sentiment(yelp_2)

-0.5

In [40]:
# Calculate sentiment of yelp_3.
simple_sentiment(yelp_3)

0.0

<details><summary> What are some shortcomings of this method? </summary>

- Primarily, we're limited to the positive/negative words we came up with.
- If someone wrote "not good" or "not bad," our sentiment function would probably treat "not good" as positive or neutral... but it's probably supposed to mean negative!
- The ordering of the words doesn't matter here, which is not how language generally works.
- We haven't corrected for misspellings.
</details>

There are a couple of ways to proceed with sentiment analysis:

1. If you have already-labeled data, you can build a supervised learning model.
2. If you don't have labeled data, you can use a Lexicon that has already been built/trained for sentiment analysis.
    - There are a bunch of these and which to use depends on your purpose/data. Here are just a few that are available:
        - AFINN lexicon
        - MPQA subjectivity lexicon
        - SentiWordNet
        - VADER lexicon

We will use the VADER (Valence Aware Dictionary and sEntiment Reasoner) lexicon to analyze the sentiments of our reviews!

In [41]:
# Instantiate Sentiment Intensity Analyzer
sa = SentimentIntensityAnalyzer()

In [42]:
# Calculate sentiment of yelp_1.
sa.polarity_scores(yelp_1)

{'neg': 0.046, 'neu': 0.799, 'pos': 0.155, 'compound': 0.9795}

In [43]:
# Calculate sentiment of yelp_2.
sa.polarity_scores(yelp_2)

{'neg': 0.078, 'neu': 0.875, 'pos': 0.047, 'compound': -0.5847}

In [44]:
# Calculate sentiment of yelp_3.
sa.polarity_scores(yelp_3)

{'neg': 0.068, 'neu': 0.902, 'pos': 0.03, 'compound': -0.7351}

In [45]:
sa.polarity_scores('this is awesome')

{'neg': 0.0, 'neu': 0.328, 'pos': 0.672, 'compound': 0.6249}

In [46]:
sa.polarity_scores('this is AWESOME')

{'neg': 0.0, 'neu': 0.293, 'pos': 0.707, 'compound': 0.7034}

In [47]:
sa.polarity_scores(':)')

{'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.4588}

In [48]:
sa.polarity_scores(':(')

{'neg': 1.0, 'neu': 0.0, 'pos': 0.0, 'compound': -0.4404}

Let's try analyzing the sentiment of IMDb movie reviews. The data is from [Kaggle](https://www.kaggle.com/c/word2vec-nlp-tutorial#part-1-for-beginners-bag-of-words).

In [49]:
# Read in training data.
reviews = pd.read_csv("../data/labeledTrainData.tsv", header=0, delimiter="\t", quoting=3)

In [50]:
# View the first five rows.
reviews.head()

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."
3,"""3630_4""",0,"""It must be assumed that those who praised thi..."
4,"""9495_8""",1,"""Superbly trashy and wondrously unpretentious ..."


In [51]:
# Examine a review.
reviews['review'][2]

'"The film starts with a manager (Nicholas Bell) giving welcome investors (Robert Carradine) to Primal Park . A secret project mutating a primal animal using fossilized DNA, like ¨Jurassik Park¨, and some scientists resurrect one of nature\'s most fearsome predators, the Sabretooth tiger or Smilodon . Scientific ambition turns deadly, however, and when the high voltage fence is opened the creature escape and begins savagely stalking its prey - the human visitors , tourists and scientific.Meanwhile some youngsters enter in the restricted area of the security center and are attacked by a pack of large pre-historical animals which are deadlier and bigger . In addition , a security agent (Stacy Haiduk) and her mate (Brian Wimmer) fight hardly against the carnivorous Smilodons. The Sabretooths, themselves , of course, are the real star stars and they are astounding terrifyingly though not convincing. The giant animals savagely are stalking its prey and the group run afoul and fight against 

In [52]:
# Sentiment of the review.
sa.polarity_scores(reviews['review'][2])

{'neg': 0.142, 'neu': 0.8, 'pos': 0.058, 'compound': -0.9883}

In [53]:
# Does this match the sentiment given in the training data?
reviews['sentiment'][2]

0

In [54]:
reviews_short = reviews.copy().head(50)

In [55]:
reviews_short['vader_sent'] = reviews_short.review.apply(lambda r: sa.polarity_scores(r)['compound'])

In [56]:
reviews_short.head()

,id,sentiment,review,vader_sent
0,"""5814_8""",1,"""With all this stuff going down at the moment ...",-0.8278
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ...",0.9819
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell...",-0.9883
3,"""3630_4""",0,"""It must be assumed that those who praised thi...",-0.2189
4,"""9495_8""",1,"""Superbly trashy and wondrously unpretentious ...",0.7960


## Interview Question

<details><summary>When processing text, what can you do about frequently occurring words, like "a," "and," "the," etc.?</summary>

- These words, called "stopwords," can either be kept or removed.
    - If we think these words do help explain our $Y$ variable, we might keep them. (For example, if we're classifying the era of a poem, the frequency of the word "the" may be helpful information!)
    - If we think these words don't help explain our $Y$ variable, we might remove them. (For example, in sentiment analysis, we might not think that people who use "the" more or less frequently are happier or angrier.)
</details>

A couple things to note:
1. NLP broadly describes: 
    - how we can get unstructured text data into a more structured form that can be interpreted by computers, and 
    - algorithms for interpreting text data.
2. That does not mean these tools we used today work to the exclusion of other methods. You can and should include other variables in your model!
    - For example, maybe the length of a review tells us something about how much people liked/disliked the movie, or maybe additional information about the reviewer (i.e. geography, age, how many reviews they had submitted) has predictive value.